# Figure: Vapor composition during degassing
Calculated values of C/S and log(SO2/H2S) molar ratios in the vapor phase as a function of normalized pressure of degassing for each volcanic system.

In [ ]:
from pathlib import Path

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from helpers.plot_styles import (
    PLOTLY_TICK_LEN,
    PLOTLY_FONT,
    PLOTLY_LEGEND_FONTSIZE,
    SAMPLE_DISPLAY_NAMES,
    TOOL_COLORS_HEX,
)
from helpers.degassing_data import load_all_systems


# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]
TOOLS  = ["DCompress", "DCompress (IM)", "EVo", "MAGEC", "SulfurX", "VolFe"]

systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)

# calculate log(SO2/H2O) molar in the vapor
for by_tool in systems.values():
    for df in by_tool.values():
        df["log_SO2_H2S_ratio"] = np.log(
            df["SO2_v_mf"] / df["H2S_v_mf"].where(df["H2S_v_mf"] > 0)
        )


## Helper to build line plot

In [ ]:
def _add_sample_lines(fig, row, col, sample, col_name="CS_v_mf", showlegend_tools=False):
    """Draw one sample's per-tool lines for ``col_name`` into a subplot."""
    for tool in TOOLS:
        df = systems.get(sample, {}).get(tool)
        if df is None or col_name not in df.columns or "P_bars" not in df.columns:
            continue
        p_init = df["P_bars"].iloc[0]
        if p_init == 0:
            continue
        x_norm = df["P_bars"] / p_init
        fig.add_trace(
            go.Scatter(
                mode="lines",
                x=x_norm, y=df[col_name],
                name=tool,
                line=dict(color=TOOL_COLORS_HEX.get(tool, "#333"), width=2),
                legendgroup=f"tool-{tool}",
                showlegend=showlegend_tools,
            ),
            row=row, col=col,
        )


## Build the figure

In [ ]:
figure_column_titles = (
    [SAMPLE_DISPLAY_NAMES.get(s, s) for s in SAMPLES]
    + [""] * 5
)

fig = make_subplots(
    rows=2, cols=4,
    shared_xaxes=True,
    horizontal_spacing=0.04, vertical_spacing=0.06,
    subplot_titles=figure_column_titles,
)

# Row 1: per-tool C/S vapor lines
for c, sample in enumerate(SAMPLES, start=1):
    _add_sample_lines(fig, row=1, col=c, sample=sample)
    fig.update_yaxes(title="C/S", row=1, col=1)
    fig.update_yaxes(range=[0, 150], row=1, col=c)

# Row 2: log(SO2/H2S) vapor ratio
for c, sample in enumerate(SAMPLES, start=1):
    _add_sample_lines(fig, row=2, col=c, sample=sample,
                      col_name="log_SO2_H2S_ratio",
                      showlegend_tools=(c == 4))
    fig.update_xaxes(title_text="P / P<sub>i</sub>",
                            range=[0, 1], row=2, col=c)
    fig.update_yaxes(title="log(SO<sub>2</sub>/H<sub>2</sub>S)", row=2, col=1)
    fig.update_yaxes(range=[-4.5, 10], row=2, col=c)

morb = fig.get_subplot(2, 1)

_legend_base = dict(
    font=dict(size=PLOTLY_LEGEND_FONTSIZE),
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
    xanchor="right",
    yanchor="top",
    tracegroupgap=0,
)
legend_models = dict(
    _legend_base,
    x=morb.xaxis.domain[1] - 0.005,
    y=morb.yaxis.domain[0] + 0.455,
)

fig.update_layout(
    height=640, width=1100,
    plot_bgcolor="white",
    font=PLOTLY_FONT,
    legend=legend_models,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
)

if SAVE_FIG:
    fig.write_image("figures/Fig_Pnorm_vapor_species.png", scale=4)

fig.show()
